# Original Dataset Overview

This notebook displays the original PhysioNet-style PSV files before preprocessing. It is for project demonstration and inspection only; it does not modify the dataset or production pipeline.

In [14]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# This notebook is inside the project's notebooks/ folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
SET_A = RAW_DIR / 'training_setA'
SET_B = RAW_DIR / 'training_setB'

print('Project root:', PROJECT_ROOT.resolve())
print('Training set A exists:', SET_A.exists())
print('Training set B exists:', SET_B.exists())

Project root: E:\Earlysepsispredicton
Training set A exists: True
Training set B exists: True


## 1. Display one original patient file

Each PSV file contains one patient's chronological ICU observations. The pipe character (`|`) is the field separator.

In [15]:
patient_file = SET_A / 'p000002.psv'

if not patient_file.exists():
    patient_files = sorted(SET_A.glob('*.psv'))
    if not patient_files:
        raise FileNotFoundError(f'No PSV files found in {SET_A}')
    patient_file = patient_files[0]

patient_data = pd.read_csv(patient_file, sep='|')
print('File:', patient_file)
print('Rows and columns:', patient_data.shape)
display(patient_data.head(10))

File: e:\Earlysepsispredicton\data\raw\training_setA\p000002.psv
Rows and columns: (23, 41)


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,75.91,0,0,1,-98.6,1,0
1,61.0,99.0,36.44,124.0,65.0,43.0,17.5,NaN,NaN,NaN,...,NaN,NaN,NaN,75.91,0,0,1,-98.6,2,0
2,64.0,98.0,NaN,125.0,64.0,41.0,27.0,NaN,NaN,NaN,...,NaN,NaN,NaN,75.91,0,0,1,-98.6,3,0
3,56.0,100.0,NaN,123.0,65.0,41.0,9.0,NaN,NaN,NaN,...,NaN,NaN,NaN,75.91,0,0,1,-98.6,4,0
4,66.0,99.0,NaN,120.0,67.0,43.0,23.0,NaN,NaN,NaN,...,NaN,NaN,NaN,75.91,0,0,1,-98.6,5,0
5,94.0,100.0,36.22,194.0,116.0,66.0,14.0,NaN,NaN,NaN,...,NaN,NaN,NaN,75.91,0,0,1,-98.6,6,0
6,58.0,99.0,NaN,133.0,68.0,43.0,13.0,NaN,NaN,NaN,...,NaN,NaN,NaN,75.91,0,0,1,-98.6,7,0
7,57.0,100.0,NaN,118.0,62.0,37.0,18.0,NaN,NaN,NaN,...,NaN,NaN,NaN,75.91,0,0,1,-98.6,8,0
8,62.0,100.0,NaN,126.0,66.0,37.0,12.0,NaN,NaN,NaN,...,NaN,NaN,NaN,75.91,0,0,1,-98.6,9,0
9,58.0,95.0,36.11,143.0,77.0,47.0,11.0,NaN,NaN,22.0,...,11.0,NaN,158.0,75.91,0,0,1,-98.6,10,0


## 2. Chronological ICU observations and labels

In [16]:
display(patient_data[['ICULOS', 'SepsisLabel']].head(15))

if 'SepsisLabel' in patient_data.columns:
    print('Patient label counts:')
    display(patient_data['SepsisLabel'].value_counts(dropna=False).rename_axis('SepsisLabel').to_frame('observations'))

,ICULOS,SepsisLabel
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0
5,6,0
6,7,0
7,8,0
8,9,0
9,10,0


Patient label counts:


,observations
SepsisLabel,
0,23


## 3. Dataset columns

In [17]:
columns = pd.DataFrame({
    'column_number': range(1, len(patient_data.columns) + 1),
    'column_name': patient_data.columns,
    'data_type': patient_data.dtypes.astype(str).values,
})
display(columns)

,column_number,column_name,data_type
0,1,HR,float64
1,2,O2Sat,float64
2,3,Temp,float64
3,4,SBP,float64
4,5,MAP,float64
5,6,DBP,float64
6,7,Resp,float64
7,8,EtCO2,float64
8,9,BaseExcess,float64
9,10,HCO3,float64


## 4. Missing-value summary for the displayed patient

In [18]:
missing_summary = pd.DataFrame({
    'missing_count': patient_data.isna().sum(),
    'missing_percent': (patient_data.isna().mean() * 100).round(2),
}).sort_values('missing_count', ascending=False)
display(missing_summary.head(15))

,missing_count,missing_percent
EtCO2,23,100.00
BaseExcess,23,100.00
Lactate,23,100.00
AST,23,100.00
Alkalinephos,23,100.00
SaO2,23,100.00
PaCO2,23,100.00
pH,23,100.00
FiO2,23,100.00
PTT,23,100.00


## Meaning of `NaN` and missing-value handling

`NaN` means **Not a Number** and represents a missing or unavailable measurement. For example, a laboratory test may not have been recorded during a particular ICU hour. `NaN` does not mean that the measurement was zero and does not automatically indicate that the patient had no clinical problem.

During preprocessing, the project handles missing values in two ways:

1. A missing-value indicator records that the measurement was absent.
2. The missing numeric value is filled using a median calculated from the training data only.

This allows the model to process the row while retaining information about whether the test was measured.

In [19]:
# Display missing values clearly for the review
nan_view = patient_data.isna().sum().sort_values(ascending=False)
nan_view = nan_view[nan_view > 0].rename('missing_count').to_frame()
nan_view['meaning'] = 'Measurement was not recorded or unavailable for some ICU observations'
display(nan_view.head(15))

,missing_count,meaning
EtCO2,23,Measurement was not recorded or unavailable fo...
BaseExcess,23,Measurement was not recorded or unavailable fo...
Lactate,23,Measurement was not recorded or unavailable fo...
AST,23,Measurement was not recorded or unavailable fo...
Alkalinephos,23,Measurement was not recorded or unavailable fo...
SaO2,23,Measurement was not recorded or unavailable fo...
PaCO2,23,Measurement was not recorded or unavailable fo...
pH,23,Measurement was not recorded or unavailable fo...
FiO2,23,Measurement was not recorded or unavailable fo...
PTT,23,Measurement was not recorded or unavailable fo...


## 5. Number of original patient files

This counts patient files without loading every file into memory.

In [20]:
files_a = sorted(SET_A.glob('*.psv'))
files_b = sorted(SET_B.glob('*.psv'))
print('Training set A patient files:', len(files_a))
print('Training set B patient files:', len(files_b))
print('Total original patient files:', len(files_a) + len(files_b))

Training set A patient files: 20336
Training set B patient files: 20000
Total original patient files: 40336


## Review explanation

The original dataset contains one pipe-separated PSV file per patient. Each row represents an ICU observation hour. The columns include vital signs, laboratory measurements, demographic/context variables, ICU hour (`ICULOS`), and the target label (`SepsisLabel`). Missing values are visible in this original view and are handled later by the preprocessing pipeline.